In [ ]:
import os
import time
import pickle
import warnings

import gdown
import librosa
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l12/genres.zip'
PICKLE_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l12/audio_data_mean.pickle'

gdown.download(DATA_URL, None, quiet=False)
gdown.download(PICKLE_URL, None, quiet=True)

!unzip -qo genres.zip

FILE_DIR = './genres'
CLASS_LIST = sorted(os.listdir(FILE_DIR))
CLASS_COUNT = len(CLASS_LIST)

CLASS_FILES = 100
FILE_INDEX_TRAIN_SPLIT = 90
VALIDATION_SPLIT = 0.1
DURATION_SEC = 30
N_FFT = 8192
HOP_LENGTH = 512
BATCH_SIZE = 32
EPOCHS = 1000
LEARNING_RATE = 0.0001

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Список классов:', CLASS_LIST)
print('Устройство:', DEVICE)

In [ ]:
def to_ohe(label_index, class_count):
    vector = np.zeros(class_count, dtype='float32')
    vector[label_index] = 1.0
    return vector


def take_audio_features(signal, sample_rate, n_fft=N_FFT, hop_length=HOP_LENGTH):
    chroma = librosa.feature.chroma_stft(y=signal, sr=sample_rate, n_fft=n_fft, hop_length=hop_length)
    mfcc = librosa.feature.mfcc(y=signal, sr=sample_rate, n_fft=n_fft, hop_length=hop_length)
    rms = librosa.feature.rms(y=signal, hop_length=hop_length)
    centroid = librosa.feature.spectral_centroid(y=signal, sr=sample_rate, n_fft=n_fft, hop_length=hop_length)
    bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=sample_rate, n_fft=n_fft, hop_length=hop_length)
    rolloff = librosa.feature.spectral_rolloff(y=signal, sr=sample_rate, n_fft=n_fft, hop_length=hop_length)
    zero_crossing = librosa.feature.zero_crossing_rate(signal, hop_length=hop_length)

    return {
        'rms': rms.mean(axis=1, keepdims=True),
        'centroid': centroid.mean(axis=1, keepdims=True),
        'bandwidth': bandwidth.mean(axis=1, keepdims=True),
        'rolloff': rolloff.mean(axis=1, keepdims=True),
        'zero_crossing': zero_crossing.mean(axis=1, keepdims=True),
        'mfcc': mfcc.mean(axis=1, keepdims=True),
        'chroma': chroma.mean(axis=1, keepdims=True)
    }


def join_feature_blocks(feature_dict):
    result = None
    for value in feature_dict.values():
        result = value if result is None else np.vstack((result, value))
    return result.T


def make_file_name(class_index, file_index):
    genre = CLASS_LIST[class_index]
    return f'{FILE_DIR}/{genre}/{genre}.{str(file_index).zfill(5)}.au'


def get_file_vectors(class_index, file_path, duration_sec):
    signal, sample_rate = librosa.load(file_path, mono=True, duration=duration_sec)
    feature_row = join_feature_blocks(take_audio_features(signal, sample_rate))
    label_vector = to_ohe(class_index, CLASS_COUNT)
    return feature_row, label_vector


def process_file(class_index, file_index, duration_sec):
    file_path = make_file_name(class_index, file_index)
    feature_row, label_vector = get_file_vectors(class_index, file_path, duration_sec)
    x_items = []
    y_items = []

    for row_index in range(feature_row.shape[0]):
        x_items.append(feature_row[row_index])
        y_items.append(label_vector)

    return file_path, np.array(x_items, dtype='float32'), np.array(y_items, dtype='float32')


def extract_data(first_file, last_file, duration_sec=DURATION_SEC):
    x_data = None
    y_data = None
    start_time = time.time()

    for class_index, genre_name in enumerate(CLASS_LIST):
        for file_index in range(first_file, last_file):
            _, current_x, current_y = process_file(class_index, file_index, duration_sec)
            x_data = current_x if x_data is None else np.vstack((x_data, current_x))
            y_data = current_y if y_data is None else np.vstack((y_data, current_y))

        print(f'Жанр {genre_name} готов -> {round(time.time() - start_time)} c')
        start_time = time.time()

    return x_data, y_data

In [ ]:
pickle_path = 'audio_data_mean.pickle'
if not os.path.exists(pickle_path):
    pickle_path = '/content/audio_data_mean.pickle'

with open(pickle_path, 'rb') as file:
    x_train_data, y_train_data = pickle.load(file)

x_scaler = StandardScaler()
x_train_data_scaled = x_scaler.fit_transform(x_train_data)

y_labels = np.argmax(y_train_data, axis=1).astype('int64') if np.asarray(y_train_data).ndim > 1 else np.asarray(y_train_data).astype('int64')

x_train, x_val, y_train, y_val = train_test_split(
    x_train_data_scaled,
    y_labels,
    stratify=y_labels,
    test_size=VALIDATION_SPLIT
)

train_dataset = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

val_dataset = TensorDataset(
    torch.tensor(x_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def plot_training_result(history):
    fig, (accuracy_axis, loss_axis) = plt.subplots(1, 2, figsize=(20, 5))
    fig.suptitle('Ход обучения модели')

    accuracy_axis.plot(history['accuracy'], label='Обучающая выборка')
    accuracy_axis.plot(history['val_accuracy'], label='Проверочная выборка')
    accuracy_axis.xaxis.get_major_locator().set_params(integer=True)
    accuracy_axis.set_xlabel('Эпоха')
    accuracy_axis.set_ylabel('Точность')
    accuracy_axis.legend()

    loss_axis.plot(history['loss'], label='Обучающая выборка')
    loss_axis.plot(history['val_loss'], label='Проверочная выборка')
    loss_axis.xaxis.get_major_locator().set_params(integer=True)
    loss_axis.set_xlabel('Эпоха')
    loss_axis.set_ylabel('Ошибка')
    loss_axis.legend()

    plt.show()


def get_probabilities(model, x_array):
    model.eval()
    x_tensor = torch.tensor(x_array, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        logits = model(x_tensor)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

    return probabilities


def classify_file(model, x_scaler, class_index, file_index):
    file_path, file_x_data, _ = process_file(class_index, file_index, DURATION_SEC)
    file_x_data = x_scaler.transform(file_x_data)

    print('Файл:', file_path)
    print('Векторы для предсказания:', file_x_data.shape)

    predictions = get_probabilities(model, file_x_data)
    mean_prediction = predictions.mean(axis=0)
    predicted_class = np.argmax(mean_prediction)
    correct = predicted_class == class_index

    print('Классификация сети:', CLASS_LIST[predicted_class], '-', 'ВЕРНО :-)' if correct else 'НЕВЕРНО.')

    plt.figure(figsize=(10, 3))
    plt.title('Среднее распределение векторов предсказаний')
    plt.bar(CLASS_LIST, mean_prediction, color='g' if correct else 'r')
    plt.show()
    print('---------------------------------------------------------------')

    return predicted_class


def classify_test_files(model, x_scaler, from_index, n_files):
    total = 0
    correct = 0
    y_true = []
    y_pred = []

    for class_index in range(CLASS_COUNT):
        for file_index in range(from_index, from_index + n_files):
            predicted_class = classify_file(model, x_scaler, class_index, file_index)
            y_true.append(class_index)
            y_pred.append(predicted_class)
            total += 1
            correct += int(predicted_class == class_index)

    accuracy_percent = round(correct / total * 100., 2)
    print(f'=== Обработано образцов: {total}, из них распознано верно: {correct}, доля верных: {accuracy_percent}% ===')

    matrix = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title('Матрица ошибок по файлам аудио')
    display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=CLASS_LIST)
    display.plot(ax=ax)
    plt.show()

# **Задание 1. Проверить форму данных обучающей и проверочной выборок**

In [ ]:
print('x_train:', x_train.shape)
print('y_train:', y_train.shape)
print('x_val:', x_val.shape)
print('y_val:', y_val.shape)

# **Задание 2. Составить модель классификатора на полносвязных слоях**

In [ ]:
class DenseGenreClassifier(nn.Module):
    def __init__(self, input_size, class_count):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.BatchNorm1d(16),
            nn.Linear(16, class_count)
        )

    def forward(self, x):
        return self.layers(x)


model = DenseGenreClassifier(
    input_size=x_train.shape[1],
    class_count=CLASS_COUNT
).to(DEVICE)

# **Задание 3. Подготовить модель к обучению**

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print('Количество параметров:', sum(parameter.numel() for parameter in model.parameters()))

# **Задание 4. Обучить модель**

In [ ]:
def run_stage(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            if is_train:
                loss.backward()
                optimizer.step()

        batch_size = x_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_count += batch_size

    return total_loss / total_count, total_correct / total_count


def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_stage(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_stage(model, val_loader, criterion)

        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)

        print(f'{epoch}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}')

    return history


history = train_model(model, train_loader, val_loader, criterion, optimizer, EPOCHS)
plot_training_result(history)

# **Задание 5. Проверить работу модели**

In [ ]:
classify_test_files(
    model=model,
    x_scaler=x_scaler,
    from_index=90,
    n_files=10
)